# Revisão humana das referências de avaliação

As 14 perguntas iniciais são rascunhos. Os nichos esperados não são prova de que um
trecho responde à pergunta. Leia os candidatos e marque IDs relevantes somente após
conferir o apoio na fonte. Incluir também exemplos fora do escopo evita aprovar um
RAG que responde a tudo.

O exemplo a seguir roda localmente, após sincronizar a Gold. As referências devem
ser independentes do retriever que será avaliado; examine candidatos de BM25,
busca densa e fontes originais para reduzir viés e falsos negativos.

In [ ]:
from pathlib import Path
import sys, json
from dotenv import load_dotenv
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src/rag_cognus").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")
from rag_cognus.config import Settings
from rag_cognus.gold import load_snapshot
from rag_cognus.retrieval import GoldRetriever
from rag_cognus.evaluation import load_cases

path = PROJECT_ROOT / "data/evaluation/questions.jsonl"
cases = load_cases(path)
documents = load_snapshot(PROJECT_ROOT / ".cache/gold.jsonl")
index = 0
case = cases[index]
print(case["inputs"]["query"])
for doc in GoldRetriever(documents, Settings()).search(case["inputs"]["query"], k=10):
    print("\nID:", doc.metadata["chunk_id"], "Fonte:", doc.metadata.get("source_url"))
    print(doc.page_content)

In [ ]:
# Preencha depois de ler os textos e conferir as fontes.
REVISADO_POR_HUMANO = False
RELEVANT_CHUNK_IDS = []
EXPECTED_FACTS = []
SHOULD_ABSTAIN = None

if REVISADO_POR_HUMANO:
    known = {d.metadata["chunk_id"] for d in documents}
    if not set(RELEVANT_CHUNK_IDS).issubset(known):
        raise ValueError("IDs ausentes do snapshot atual.")
    if not RELEVANT_CHUNK_IDS and SHOULD_ABSTAIN is not True:
        raise ValueError("Informe referências ou marque o caso como abstenção esperada.")
    case["expectations"].update(label_status="human_verified",
                                relevant_chunk_ids=RELEVANT_CHUNK_IDS,
                                expected_facts=EXPECTED_FACTS,
                                should_abstain=SHOULD_ABSTAIN)
    path.write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in cases) + "\n")
    print("Referências salvas. Revise o diff do dataset antes de compartilhar.")
else:
    print("Dataset preservado; nenhuma referência foi marcada como revisada.")